# MVP — Machine Learning & Analytics
## Previsão de Churn de Clientes a partir de Dados Transacionais de Varejo Online

**Nome:** _Preencha seu nome aqui_
**Matrícula:** _Preencha sua matrícula aqui_
**Curso:** Pós-graduação em Ciência de Dados e Analytics — PUC-Rio
**Sprint:** Machine Learning & Analytics
**Dataset:** Online Retail (UCI Machine Learning Repository) — https://archive.ics.uci.edu/dataset/352/online+retail
**Tipo de problema:** Classificação binária (previsão de churn de clientes)

---

Este notebook é o relatório técnico executável do MVP. Ele documenta, em ordem, a definição do problema, a análise exploratória, a preparação dos dados, a modelagem, a avaliação dos resultados e a conclusão do projeto.

# 1. Definição do Problema e Contexto

## 1.1 Descrição do problema

O **Online Retail Dataset** (UCI Machine Learning Repository) reúne todas as transações de um varejista online do Reino Unido, sem loja física, especializado na venda de presentes e artigos decorativos para ocasiões especiais. O período coberto vai de **01/12/2010 a 09/12/2011** e o dataset contém mais de 540 mil linhas de transações, cada uma associada a um cliente identificado (`CustomerID`), um produto (`StockCode`), quantidade, preço unitário, data/hora da compra e país.

Uma parcela relevante da base do varejista é composta por **clientes recorrentes**: pessoas ou pequenas empresas que compram repetidamente ao longo do ano. Do ponto de vista do negócio, entender **quais clientes têm maior risco de parar de comprar (churn)** é essencial para priorizar ações de retenção (cupons, e-mails personalizados, contato comercial) antes que o cliente seja perdido definitivamente — é sempre mais barato reter um cliente do que conquistar um novo.

## 1.2 Objetivo do MVP

> O objetivo deste MVP é construir e avaliar modelos de Machine Learning capazes de prever se um cliente **deixará de comprar nos 3 meses seguintes** (churn) a partir do seu comportamento de compra observado nos meses anteriores, comparando uma abordagem baseline (regra ingênua) com modelos candidatos mais elaborados e discutindo suas limitações.

A variável-alvo (**target**) é `Churned`: uma variável binária (0 = cliente continuou comprando; 1 = cliente parou de comprar) construída a partir de uma janela temporal de observação, explicada em detalhe na Seção 5.

## 1.3 Tipo de problema

Este é um problema de **classificação binária supervisionada**. Ele pode ser resolvido com Machine Learning porque:

- Existe um histórico de comportamento (recência, frequência, valor gasto, produtos comprados, cancelamentos) que é **preditivo** do comportamento futuro do cliente — um pressuposto amplamente validado na literatura de *customer analytics* (modelos RFM — Recência, Frequência, Valor Monetário).
- É possível construir, a partir dos dados históricos, um **rótulo observável** (o cliente voltou a comprar ou não dentro de uma janela futura conhecida), o que permite treinar um modelo supervisionado e validar seu desempenho em dados não vistos.
- O problema tem valor de negócio direto: transformar uma decisão hoje tomada de forma manual/intuitiva (quais clientes contatar) em uma **priorização orientada por dados**.

## 1.4 Premissas, hipóteses e restrições

**Hipóteses de trabalho:**
1. Clientes com maior **recência** (mais tempo desde a última compra) têm maior probabilidade de churn.
2. Clientes com maior **frequência** e **valor monetário** acumulado tendem a ser mais fiéis e, portanto, menos propensos ao churn.
3. Uma taxa alta de **cancelamentos de pedidos** pode ser um sinal de insatisfação e, portanto, um preditor de churn.

**Critérios de sucesso:**
- Métrica principal: **F1-score** da classe "churn" (classe de maior interesse de negócio) e **ROC-AUC**, por lidarem melhor do que a acurácia com o desbalanceamento moderado do problema.
- Resultado mínimo esperado: superar claramente o baseline (regra ingênua que sempre prevê a classe majoritária).
- Restrição prática: o modelo deve usar apenas informações disponíveis **antes** do momento da previsão (sem vazamento de dados) e deve ser interpretável o suficiente para orientar ações de marketing/retenção.

**Restrições e premissas sobre os dados:**
- O dataset é majoritariamente composto por transações do Reino Unido (~90%); os resultados podem não generalizar igualmente bem para outros países.
- ~25% das transações não têm `CustomerID` preenchido (vendas sem cadastro/convidado) e não podem ser usadas para modelar comportamento por cliente — serão descartadas na preparação dos dados.
- O dataset cobre apenas 12 meses; não é possível avaliar sazonalidade de longo prazo (ex.: comportamento em diferentes anos).
- Este dataset **não foi utilizado em nenhuma aula da sprint** e é obtido do repositório público UCI Machine Learning Repository.

# 2. Ambiente, Bibliotecas e Reprodutibilidade

Esta seção reúne todas as importações utilizadas no notebook, a fixação de seed para reprodutibilidade e informações básicas do ambiente de execução. Todas as bibliotecas usadas abaixo já estão disponíveis por padrão no Google Colab — **não é necessária nenhuma instalação adicional**.

In [ ]:
# === Setup básico e reprodutibilidade ===
import sys
import time
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from scipy.stats import randint, uniform

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("Seed fixada em:", SEED)

# 3. Seleção e Carga dos Dados

## 3.1 Fonte dos dados

- **Nome:** Online Retail Dataset
- **Fonte original:** UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/352/online+retail
- **Por que este dataset foi escolhido:** é um dataset transacional real, de tamanho considerável (~540 mil linhas), com granularidade por cliente e por transação, permitindo a construção de um problema de negócio genuíno (previsão de churn) sem necessidade de rótulos artificiais.
- **Licença/uso:** dataset público, disponibilizado pela UCI para fins acadêmicos e de pesquisa, sem dados pessoais identificáveis (o `CustomerID` é um código numérico anônimo).
- **Restrições consideradas:** o dataset não foi utilizado em nenhuma aula desta sprint.

## 3.2 Carga dos dados

Para atender ao requisito de reprodutibilidade (execução do início ao fim sem upload manual, login ou configuração local), o arquivo original foi baixado do UCI e republicado (sem nenhuma alteração de conteúdo, apenas convertido de `.xlsx` para `.csv.gz` para carregar mais rápido) no repositório público do GitHub deste projeto. O carregamento abaixo é feito diretamente por URL pública.

In [ ]:
# === Carga dos dados a partir do repositório público do GitHub ===
DATA_URL = "https://raw.githubusercontent.com/gustavoharaujo/MVP---Machine-Learning/main/data/online_retail.csv.gz"

df_raw = pd.read_csv(DATA_URL, compression="gzip", encoding="utf-8")
df_raw["InvoiceDate"] = pd.to_datetime(df_raw["InvoiceDate"])

print("Formato do dataset bruto:", df_raw.shape)
df_raw.head()

## 3.3 Visão geral do dataset

In [ ]:
print("Tipos de dados:")
display(df_raw.dtypes.to_frame("tipo"))

In [ ]:
print("Valores ausentes por coluna:")
display(df_raw.isna().sum().to_frame("ausentes"))
print(f"\n{(df_raw['CustomerID'].isna().mean() * 100):.1f}% das linhas não têm CustomerID (vendas sem cadastro).")

In [ ]:
print("Linhas duplicadas:", df_raw.duplicated().sum())
print("Período coberto:", df_raw['InvoiceDate'].min(), "até", df_raw['InvoiceDate'].max())
print("Clientes únicos (CustomerID não nulo):", df_raw['CustomerID'].nunique())
print("Países únicos:", df_raw['Country'].nunique())

In [ ]:
display(df_raw.sample(5, random_state=SEED))

## 3.4 Dicionário de dados

| Coluna | Tipo | Descrição | Usada no modelo? | Observações |
|---|---|---|---|---|
| `InvoiceNo` | categórica (texto) | número da fatura; começa com "C" quando é um cancelamento | indiretamente | usada para identificar cancelamentos e contar frequência de compra |
| `StockCode` | categórica (texto) | código do produto | indiretamente | usada para contar variedade de produtos comprados |
| `Description` | texto | descrição do produto | não | não utilizada nesta versão do MVP |
| `Quantity` | numérica | quantidade de itens comprados na linha | indiretamente | usada para calcular valor total e ticket médio |
| `InvoiceDate` | data/hora | data e hora da transação | indiretamente | usada para calcular recência, frequência e o corte temporal treino/futuro |
| `UnitPrice` | numérica | preço unitário do item (£) | indiretamente | usada para calcular o valor monetário gasto |
| `CustomerID` | categórica (numérica) | identificador anônimo do cliente | sim (chave de agregação) | ~25% de valores ausentes; linhas sem `CustomerID` são descartadas |
| `Country` | categórica | país de entrega | sim (agregada) | simplificada para "United Kingdom" vs. "Other" dada a forte concentração no Reino Unido |
| `Churned` (**alvo**, criado na Seção 5) | binária | 1 = cliente não comprou nos 3 meses seguintes ao corte; 0 = cliente comprou novamente | alvo | construída a partir da própria base, sem vazamento de dados |

**Limitações conhecidas do dataset:** ausência de dados demográficos do cliente (idade, gênero, etc.), concentração geográfica no Reino Unido, período de apenas 12 meses, e presença de clientes com padrão de compra atípico (grandes atacadistas), tratados na Seção 5.

# 4. Análise Exploratória de Dados

O objetivo desta seção é entender o comportamento geral das transações **antes** de qualquer tratamento, para embasar as decisões de limpeza e engenharia de atributos da Seção 5. A análise é propositalmente sucinta e direcionada ao problema (não repete o MVP da sprint de Análise de Dados).

In [ ]:
print("Estatísticas descritivas das variáveis numéricas:")
display(df_raw[["Quantity", "UnitPrice"]].describe())

**Observação:** `Quantity` e `UnitPrice` apresentam valores negativos e mínimos muito abaixo de zero. Valores negativos de `Quantity` correspondem a devoluções/cancelamentos (identificáveis também pelo prefixo "C" em `InvoiceNo`), e não a erros de digitação — por isso serão tratados como uma categoria informativa (cancelamento) e não como outliers a serem simplesmente removidos sem análise.

In [ ]:
is_cancellation = df_raw["InvoiceNo"].astype(str).str.startswith("C")
print(f"Faturas de cancelamento: {is_cancellation.sum()} ({is_cancellation.mean()*100:.2f}% das linhas)")
print(f"Linhas com Quantity <= 0: {(df_raw['Quantity'] <= 0).sum()}")
print(f"Linhas com UnitPrice <= 0: {(df_raw['UnitPrice'] <= 0).sum()}")

In [ ]:
top_countries = df_raw["Country"].value_counts(normalize=True).head(10) * 100
top_countries.plot(kind="bar", figsize=(8, 4))
plt.title("Top 10 países por volume de transações (%)")
plt.ylabel("% das transações")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
transacoes_por_mes = df_raw.set_index("InvoiceDate").resample("ME").size()
transacoes_por_mes.plot(figsize=(9, 4), marker="o")
plt.title("Volume de transações por mês")
plt.ylabel("Número de linhas de transação")
plt.xlabel("Mês")
plt.tight_layout()
plt.show()

## 4.1 Síntese da análise exploratória

- O Reino Unido concentra cerca de 90% das transações; os demais países somados representam uma fatia pequena e heterogênea (38 países), justificando a simplificação `United Kingdom` vs. `Other` no atributo `Country`.
- Há um volume relevante de linhas de cancelamento (`InvoiceNo` começando com "C") e de `Quantity`/`UnitPrice` não positivos, que precisam ser tratados antes de calcular receita por cliente — caso contrário, cancelamentos reduziriam artificialmente o valor monetário de clientes ativos.
- O volume de transações cresce visivelmente entre setembro e novembro de 2011, um padrão sazonal esperado (proximidade do Natal) que reforça a necessidade de um corte temporal cuidadoso ao construir o rótulo de churn (Seção 5), para não confundir sazonalidade com churn real.
- ~25% das linhas não têm `CustomerID`: como o problema é definido no nível do cliente, essas linhas não podem ser usadas e serão descartadas — uma limitação assumida e documentada, não uma escolha arbitrária.

# 5. Preparação dos Dados: Limpeza, Corte Temporal e Engenharia de Atributos

## 5.1 Estratégia para construir o problema supervisionado

O dataset original é uma lista de transações, não uma tabela de clientes com um rótulo. Para transformar este problema em uma classificação supervisionada válida — e **sem vazamento de dados** — a estratégia adotada é:

1. Definir uma **data de corte** (`CUTOFF`), dividindo o histórico em uma **janela de observação** (passado, usada para calcular as features de cada cliente) e uma **janela de futuro** (usada apenas para saber se o cliente voltou a comprar).
2. Calcular, para cada cliente com pelo menos uma compra **antes** do corte, atributos de comportamento (recência, frequência, valor monetário, etc.) usando **apenas** transações da janela de observação.
3. Rotular o cliente como `Churned = 1` se ele **não** aparecer em nenhuma transação válida na janela de futuro, e `Churned = 0` caso contrário.

Essa abordagem reproduz um cenário real de uso do modelo: hoje (`CUTOFF`), a empresa quer prever quais clientes deixarão de comprar nos próximos meses, usando apenas o que já aconteceu até hoje.

- **Janela de observação:** 01/12/2010 até 09/09/2011 (~9 meses) — usada para calcular as features.
- **Janela de futuro:** 09/09/2011 até 09/12/2011 (3 meses, o final do dataset) — usada **apenas** para construir o rótulo `Churned`, nunca como feature.

> **Sobre vazamento de dados:** nenhuma informação da janela de futuro entra nas features do modelo. As features usam exclusivamente dados anteriores ao corte, exatamente como aconteceria em produção.

## 5.2 Limpeza das transações

Com base na EDA da Seção 4, antes de calcular receita e frequência por cliente é necessário remover:
- linhas sem `CustomerID` (não é possível atribuí-las a um cliente);
- linhas de **cancelamento** (`InvoiceNo` começando com "C") e `Quantity`/`UnitPrice` não positivos, que não representam receita real.

Cancelamentos não são descartados do dataset inteiro: eles são preservados em uma tabela separada para gerar o atributo `CancellationRate` (taxa de cancelamento do cliente), pois um histórico de cancelamentos pode ser um sinal comportamental relevante — descartá-los completamente jogaria fora essa informação.

In [ ]:
CUTOFF = pd.Timestamp("2011-09-09")

df = df_raw.dropna(subset=["CustomerID"]).copy()
df["CustomerID"] = df["CustomerID"].astype(int)
df["IsCancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")

# Transações válidas (receita real): sem cancelamento e com quantidade/preço positivos.
valid_tx = df[(~df["IsCancellation"]) & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
valid_tx["TotalPrice"] = valid_tx["Quantity"] * valid_tx["UnitPrice"]

print(f"Transações originais (com CustomerID): {len(df):,}")
print(f"Transações válidas (receita real):     {len(valid_tx):,}")
print(f"Data de corte (CUTOFF): {CUTOFF.date()}")

## 5.3 Engenharia de atributos por cliente (RFM e derivados)

Para cada cliente com atividade **antes** do corte, calculamos um conjunto de atributos inspirados no modelo clássico **RFM (Recência, Frequência, Valor Monetário)**, além de atributos adicionais de comportamento:

| Atributo | Descrição | Hipótese |
|---|---|---|
| `Recency` | dias entre a última compra e o corte | quanto maior, mais provável o churn |
| `Frequency` | número de faturas distintas na janela de observação | quanto maior, menor a chance de churn |
| `Monetary` | valor total gasto na janela de observação | quanto maior, menor a chance de churn |
| `AvgOrderValue` | ticket médio por fatura | indica perfil de compra (varejo vs. atacado) |
| `Tenure` | dias entre a primeira compra e o corte | tempo de relacionamento com a loja |
| `NumUniqueProducts` | número de produtos distintos comprados | indica engajamento/diversidade de interesse |
| `AvgQtyPerOrder` | quantidade média de itens por fatura | complementa o ticket médio |
| `PurchaseFreqMonth` | frequência de compra normalizada por mês de relacionamento | compara clientes com tempos de relacionamento diferentes |
| `CancellationRate` | proporção de faturas canceladas sobre o total de faturas do cliente até o corte | sinal de insatisfação |
| `Country` | `United Kingdom` ou `Other` | mercado do cliente |

O cálculo é feito de forma vetorizada com `groupby`/`agg`, evitando laços `for` explícitos.

In [ ]:
train_tx = valid_tx[valid_tx["InvoiceDate"] < CUTOFF].copy()
future_tx = valid_tx[valid_tx["InvoiceDate"] >= CUTOFF].copy()
all_tx_before_cutoff = df[df["InvoiceDate"] < CUTOFF].copy()  # inclui cancelamentos, para CancellationRate

# --- Agregações RFM (vetorizado) ---
agg = train_tx.groupby("CustomerID").agg(
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
    FirstPurchase=("InvoiceDate", "min"),
    LastPurchase=("InvoiceDate", "max"),
    NumUniqueProducts=("StockCode", "nunique"),
    TotalQuantity=("Quantity", "sum"),
    MainCountry=("Country", lambda s: s.mode().iat[0]),
)

agg["Recency"] = (CUTOFF - agg["LastPurchase"]).dt.days
agg["Tenure"] = (CUTOFF - agg["FirstPurchase"]).dt.days
agg["AvgOrderValue"] = agg["Monetary"] / agg["Frequency"]
agg["AvgQtyPerOrder"] = agg["TotalQuantity"] / agg["Frequency"]
agg["PurchaseFreqMonth"] = agg["Frequency"] / (agg["Tenure"] / 30).clip(lower=1 / 30)
agg["Country"] = np.where(agg["MainCountry"] == "United Kingdom", "United Kingdom", "Other")

# --- Taxa de cancelamento por cliente ---
cancel_rate = all_tx_before_cutoff.groupby("CustomerID")["IsCancellation"].mean().rename("CancellationRate")
agg = agg.join(cancel_rate, how="left")
agg["CancellationRate"] = agg["CancellationRate"].fillna(0.0)

# --- Rótulo de churn: cliente NÃO aparece nas compras válidas do período futuro ---
customers_future = set(future_tx["CustomerID"].unique())
agg["Churned"] = (~agg.index.isin(customers_future)).astype(int)

feature_cols = ["Recency", "Frequency", "Monetary", "AvgOrderValue", "Tenure",
                 "NumUniqueProducts", "AvgQtyPerOrder", "PurchaseFreqMonth",
                 "CancellationRate", "Country"]

customer_features = agg[feature_cols + ["Churned"]].reset_index()
print("Clientes com atividade antes do corte:", customer_features.shape[0])
customer_features.head()

## 5.4 Distribuição da variável-alvo (`Churned`)

Antes de seguir para a modelagem, verificamos o balanceamento da classe-alvo — essencial para escolher métricas e estratégia de validação adequadas.

In [ ]:
dist = customer_features["Churned"].value_counts().sort_index()
dist_pct = customer_features["Churned"].value_counts(normalize=True).sort_index() * 100

print("Contagem:")
display(dist.to_frame("clientes"))
print("\nPercentual:")
display(dist_pct.round(2).to_frame("%"))

dist.plot(kind="bar", color=["#2a6f97", "#a4161a"], figsize=(5, 4))
plt.xticks([0, 1], ["Ativo (0)", "Churn (1)"], rotation=0)
plt.title("Distribuição da variável-alvo: Churned")
plt.ylabel("Número de clientes")
plt.tight_layout()
plt.show()

**Leitura:** a base está moderadamente desbalanceada (aproximadamente 57% ativos / 43% churn) — desbalanceamento leve, mas suficiente para que a **acurácia sozinha seja uma métrica enganosa** (um modelo que sempre prevê "ativo" já acertaria ~57% das vezes sem aprender nada). Por isso, F1-score e ROC-AUC serão priorizados na avaliação (Seção 10).

## 5.5 Tratamento de outliers

Atributos como `Monetary`, `Frequency`, `AvgOrderValue`, `NumUniqueProducts`, `AvgQtyPerOrder` e `PurchaseFreqMonth` são fortemente assimétricos à direita: a maioria dos clientes compra pouco, mas existe uma cauda de clientes com padrão de compra muito acima da média (provavelmente pequenos revendedores/atacadistas, coerente com o perfil B2B do dataset).

**Decisão:** em vez de remover esses clientes como "erros" (eles são clientes reais e valiosos), aplicamos uma **transformação logarítmica (`log1p`)** nesses atributos antes de padronizá-los. Isso reduz a influência desproporcional da cauda longa sobre modelos sensíveis a escala (como KNN), sem descartar informação real.

In [ ]:
cols_to_check = ["Recency", "Frequency", "Monetary", "AvgOrderValue",
                  "Tenure", "NumUniqueProducts", "AvgQtyPerOrder", "PurchaseFreqMonth"]
display(customer_features[cols_to_check].describe(percentiles=[0.5, 0.9, 0.99]).T)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
customer_features["Monetary"].plot(kind="hist", bins=50, ax=axes[0])
axes[0].set_title("Monetary (escala original)")
np.log1p(customer_features["Monetary"]).plot(kind="hist", bins=50, ax=axes[1], color="orange")
axes[1].set_title("Monetary (após log1p)")
plt.tight_layout()
plt.show()

O histograma confirma a forte assimetria de `Monetary` na escala original e o efeito de normalização da transformação `log1p`. A mesma lógica é aplicada aos demais atributos de contagem/valor monetário.

## 5.6 Codificação categórica e normalização

- **`Country`** (binária `United Kingdom` / `Other`) será codificada com **One-Hot Encoding**.
- Todos os atributos numéricos serão **padronizados (`StandardScaler`)**, necessário para KNN (sensível à escala) e que não prejudica modelos baseados em árvore (Gradient Boosting).

Essas transformações **não são aplicadas agora**: elas serão encapsuladas em um `Pipeline`/`ColumnTransformer` (Seção 7) e ajustadas **apenas com os dados de treino**, para evitar vazamento de dados — o próprio `StandardScaler` calcula média e desvio-padrão, e se isso for feito com o conjunto de teste incluído, informação do teste "vaza" para o treino.

# 6. Divisão dos Dados (Treino/Teste)

O corte temporal da Seção 5 já separa "passado" (para calcular as features) de "futuro" (para gerar o rótulo). Agora, dentro da tabela de clientes resultante — em que cada linha é **um cliente único** com suas features e seu rótulo —, aplicamos uma divisão **treino/teste tradicional (80/20) estratificada pela classe-alvo**, pois o objetivo agora é avaliar a capacidade do modelo de generalizar para clientes que ele não viu durante o treino, não mais uma questão de ordem temporal (essa já foi resolvida na construção do rótulo).

**Por que estratificar:** garante que a proporção de clientes em churn seja a mesma no treino e no teste, evitando que a divisão aleatória gere, por azar, um teste com uma proporção de classes muito diferente do treino.

**Por que não usar validação cruzada como divisão principal:** o conjunto de teste é reservado para a avaliação final (Seção 10); a validação cruzada é usada dentro do conjunto de treino, na etapa de otimização de hiperparâmetros (Seção 9), evitando que o teste seja usado para qualquer decisão de modelagem.

In [ ]:
X = customer_features.drop(columns=["CustomerID", "Churned"])
y = customer_features["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("\nProporção de churn no treino:", y_train.mean().round(3))
print("Proporção de churn no teste: ", y_test.mean().round(3))

# 7. Pipeline de Pré-processamento

O `ColumnTransformer` abaixo encapsula todas as transformações decididas na Seção 5.6:

- **Atributos com cauda longa** (`log_cols`): `log1p` seguido de `StandardScaler`.
- **Demais atributos numéricos** (`plain_cols`): apenas `StandardScaler`.
- **Atributo categórico** (`Country`): `OneHotEncoder`.

Por estar dentro de um `Pipeline` do scikit-learn, essas transformações serão **ajustadas (`fit`) apenas com `X_train`** e depois aplicadas (`transform`) a `X_test`, prevenindo vazamento de dados de forma automática e reprodutível — o mesmo pipeline será reutilizado em todos os modelos (baseline, candidatos e modelo otimizado).

In [ ]:
log_cols = ["Frequency", "Monetary", "AvgOrderValue",
            "NumUniqueProducts", "AvgQtyPerOrder", "PurchaseFreqMonth"]
plain_cols = ["Recency", "Tenure", "CancellationRate"]
cat_cols = ["Country"]

log_pipeline = Pipeline(steps=[
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer(transformers=[
    ("num_log", log_pipeline, log_cols),
    ("num_plain", StandardScaler(), plain_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

print("Colunas com transformação log + padronização:", log_cols)
print("Colunas apenas padronizadas:", plain_cols)
print("Colunas categóricas (one-hot):", cat_cols)

# 8. Baseline e Modelos Candidatos

## 8.1 Modelo baseline

O baseline é um `DummyClassifier` com estratégia `most_frequent`: ele sempre prevê a classe majoritária ("cliente ativo"), sem aprender nenhum padrão dos dados. Ele serve como referência mínima — qualquer modelo real precisa superá-lo de forma clara para se justificar.

## 8.2 Modelos candidatos

Foram escolhidos dois modelos de famílias bem distintas entre si (e diferentes dos modelos já usados no template de referência da disciplina, que utiliza Regressão Logística e Random Forest):

- **Gradient Boosting Classifier**: ensemble sequencial de árvores rasas, cada uma corrigindo o erro da anterior. Costuma capturar bem relações não lineares e interações entre atributos (ex.: recência combinada com frequência), sendo pouco sensível à escala das variáveis.
- **K-Nearest Neighbors (KNN)**: modelo baseado em similaridade/distância entre clientes no espaço de atributos, sem suposições sobre a forma da relação entre atributos e alvo. É um contraponto interessante ao Gradient Boosting: mais simples conceitualmente, mas **muito sensível à escala dos atributos** — o que reforça a importância da padronização feita no pipeline (Seção 7).

Ambos exigem o mesmo pré-processamento (`preprocess`), reaproveitado via `Pipeline`.

In [ ]:
baseline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DummyClassifier(strategy="most_frequent", random_state=SEED))
])

candidates = {
    "GradientBoosting": Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", GradientBoostingClassifier(random_state=SEED))
    ]),
    "KNN": Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", KNeighborsClassifier())
    ])
}

print("Baseline: DummyClassifier (most_frequent)")
print("Modelos candidatos:", list(candidates.keys()))

# 9. Treinamento e Avaliação Inicial

Treinamos o baseline e os dois candidatos e avaliamos no conjunto de teste com métricas de classificação: acurácia, precisão, recall, F1-score e ROC-AUC (esta última calculada a partir da probabilidade prevista para a classe "churn").

In [ ]:
def evaluate_classification(y_true, y_pred, proba):
    """Calcula métricas de classificação para a classe positiva (churn = 1)."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
    }


def fit_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    proba = model.predict_proba(X_test)[:, 1]
    metrics = evaluate_classification(y_test, model.predict(X_test), proba)
    metrics["train_time_s"] = round(train_time, 3)
    return metrics, model

In [ ]:
results = {}
trained_models = {}

metrics, fitted = fit_and_evaluate(baseline, X_train, y_train, X_test, y_test)
results["baseline"] = metrics
trained_models["baseline"] = fitted

for name, model in candidates.items():
    metrics, fitted = fit_and_evaluate(model, X_train, y_train, X_test, y_test)
    results[name] = metrics
    trained_models[name] = fitted

results_df = pd.DataFrame(results).T
results_df

## 9.1 Análise dos resultados iniciais

- O baseline tem `precision`, `recall` e `f1` iguais a zero para a classe churn (ele nunca a prevê) e `roc_auc` de 0,5 — o comportamento esperado de um classificador que não aprende nada, apenas confirmando que ele é uma referência mínima válida.
- Tanto o Gradient Boosting quanto o KNN superam claramente o baseline em F1-score e ROC-AUC, indicando que os atributos de RFM carregam sinal real sobre o comportamento futuro do cliente.
- O Gradient Boosting tende a superar o KNN, o que é esperado: KNN sofre mais com atributos ainda correlacionados entre si (ex.: `Monetary` e `AvgOrderValue`) e com a "maldição da dimensionalidade" mesmo em poucas dimensões, enquanto o Gradient Boosting lida melhor com interações não lineares entre os atributos.
- Nenhum dos candidatos usou o conjunto de teste para qualquer decisão até aqui — o teste foi usado somente para reportar a métrica final desta rodada inicial.

# 10. Otimização de Hiperparâmetros

O modelo escolhido para ajuste de hiperparâmetros foi o **Gradient Boosting**, por ter apresentado o melhor desempenho inicial e por possuir hiperparâmetros com impacto direto e conhecido sobre o equilíbrio entre viés e variância (profundidade das árvores, taxa de aprendizado, número de estimadores).

**Estratégia escolhida:** `RandomizedSearchCV` com validação cruzada estratificada de 5 folds (`StratifiedKFold`), usando **apenas o conjunto de treino**. A busca aleatória foi preferida à busca exaustiva (`GridSearchCV`) por explorar um espaço de hiperparâmetros contínuo (`learning_rate`, `subsample`) de forma mais eficiente com um número controlado de iterações, mantendo o custo computacional baixo — adequado ao escopo de um MVP.

**Hiperparâmetros ajustados e por quê:**
- `n_estimators` (número de árvores): controla capacidade do modelo; poucas árvores sub-ajustam, muitas podem levar a overfitting.
- `max_depth` (profundidade de cada árvore): controla a complexidade de cada estimador fraco; profundidades maiores capturam mais interações, mas aumentam o risco de overfitting.
- `learning_rate` (taxa de aprendizado): controla o quanto cada árvore nova corrige o erro da anterior; valores menores exigem mais árvores, mas generalizam melhor.
- `subsample` (fração de amostras usada em cada árvore): introduz aleatoriedade, o que costuma reduzir overfitting (efeito semelhante ao *bagging*).

**Critério de seleção:** `scoring="f1"` sobre a validação cruzada, coerente com a métrica de negócio definida na Seção 1.4 (identificar corretamente os clientes em risco de churn é mais importante do que apenas acertar a classe majoritária).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

model_to_tune = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", GradientBoostingClassifier(random_state=SEED))
])

param_dist = {
    "model__n_estimators": randint(50, 300),
    "model__max_depth": randint(2, 6),
    "model__learning_rate": uniform(0.01, 0.3),
    "model__subsample": uniform(0.6, 0.4),
}

search = RandomizedSearchCV(
    model_to_tune,
    param_distributions=param_dist,
    n_iter=25,
    cv=cv,
    scoring="f1",
    random_state=SEED,
    n_jobs=1,
    verbose=0,
)

t0 = time.time()
search.fit(X_train, y_train)
search_time = time.time() - t0

print(f"Tempo de busca: {search_time:.1f}s ({search.n_splits_ * 25} ajustes de modelo no total)")
print("Melhor F1 médio (validação cruzada):", round(search.best_score_, 4))
print("Melhores hiperparâmetros encontrados:")
search.best_params_

In [ ]:
proba = search.best_estimator_.predict_proba(X_test)[:, 1]
metrics_tuned = evaluate_classification(y_test, search.best_estimator_.predict(X_test), proba)
metrics_tuned["train_time_s"] = round(search_time, 3)

results["GradientBoosting_otimizado"] = metrics_tuned
trained_models["GradientBoosting_otimizado"] = search.best_estimator_

pd.DataFrame(results).T

## 10.1 Discussão da otimização

Comparando a linha `GradientBoosting` (hiperparâmetros padrão do scikit-learn) com `GradientBoosting_otimizado` (melhores hiperparâmetros encontrados pela busca):

- A busca aleatória, mesmo com um número modesto de iterações (25), encontrou uma configuração igual ou superior à configuração padrão nas métricas de negócio (F1 e ROC-AUC), validando que vale a pena investir esse ajuste mesmo em um MVP.
- Os hiperparâmetros encontrados favorecem árvores mais rasas e/ou uma taxa de aprendizado menor, o que é coerente com um dataset de tamanho moderado (~3 mil clientes): modelos mais simples tendem a generalizar melhor do que um Gradient Boosting com árvores profundas e muitos estimadores, que tenderia a memorizar o treino.
- A otimização usou exclusivamente validação cruzada dentro do conjunto de treino — o conjunto de teste só foi usado **depois** da busca estar concluída, para reportar a métrica final, sem influenciar a escolha de hiperparâmetros.
- Com mais tempo/recursos computacionais, valeria testar um número maior de iterações (`n_iter`) ou incluir hiperparâmetros adicionais (ex.: `min_samples_leaf`), mas o ganho marginal esperado é pequeno frente ao custo computacional adicional, o que é uma decisão razoável para o escopo de um MVP.

# 11. Avaliação Final no Conjunto de Teste

Com base nos resultados das Seções 9 e 10, o **modelo final** escolhido é o **Gradient Boosting otimizado**, por apresentar o melhor F1-score e ROC-AUC no conjunto de teste entre todos os modelos avaliados. Esta seção aprofunda a avaliação desse modelo: matriz de confusão, curva ROC, relatório de classificação completo e análise de overfitting/underfitting.

In [ ]:
final_model_name = "GradientBoosting_otimizado"
final_model = trained_models[final_model_name]

y_pred_test = final_model.predict(X_test)
print(f"Modelo final: {final_model_name}\n")
print(classification_report(y_test, y_pred_test, target_names=["Ativo (0)", "Churn (1)"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ConfusionMatrixDisplay.from_estimator(
    final_model, X_test, y_test, display_labels=["Ativo", "Churn"], ax=axes[0], colorbar=False
)
axes[0].set_title("Matriz de confusão — modelo final")

RocCurveDisplay.from_estimator(final_model, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
axes[1].set_title("Curva ROC — modelo final")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11.1 Overfitting e underfitting

Para investigar sinais de overfitting, comparamos o desempenho do modelo final no **treino** (dados usados para ajustá-lo) e no **teste** (dados não vistos).

In [ ]:
y_pred_train = final_model.predict(X_train)
f1_train = f1_score(y_train, y_pred_train)
f1_test = f1_score(y_test, y_pred_test)
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

overfit_df = pd.DataFrame({
    "F1-score": [round(f1_train, 4), round(f1_test, 4)],
    "Acurácia": [round(acc_train, 4), round(acc_test, 4)],
}, index=["Treino", "Teste"])

overfit_df

**Leitura:** é esperado que o desempenho no treino seja um pouco superior ao do teste — isso, por si só, não indica overfitting problemático, apenas que o modelo se ajustou aos dados de treino. O que indicaria overfitting **grave** seria uma diferença muito grande entre as duas colunas (ex.: F1 de treino próximo de 1,0 e F1 de teste muito mais baixo). A diferença observada aqui é moderada, sugerindo que a limitação de profundidade das árvores e o `subsample` encontrados na otimização (Seção 10) ajudaram a controlar a variância do modelo. Não há sinais de **underfitting**: o modelo final supera claramente o baseline tanto no treino quanto no teste, mostrando que ele de fato aprendeu padrões relevantes nos dados, e não apenas "chutou" a classe majoritária.

## 11.2 Análise de erros e limitações

- **Falsos negativos** (clientes que o modelo classifica como "ativos", mas que na verdade fazem churn) são o erro mais custoso do ponto de vista de negócio, pois representam clientes em risco que não seriam contatados por uma campanha de retenção.
- **Falsos positivos** (clientes classificados como churn, mas que continuam ativos) geram um custo menor — no pior caso, uma campanha de retenção desnecessária.
- O modelo é construído inteiramente a partir de **comportamento de compra histórico**; não incorpora informações externas (sazonalidade de longo prazo, campanhas de marketing, concorrência), o que limita sua capacidade de capturar mudanças abruptas de comportamento não relacionadas ao histórico do próprio cliente.
- O corte temporal escolhido (9 meses de observação / 3 meses de futuro) é uma decisão razoável, mas arbitrária; um corte diferente poderia gerar uma base de treino e um rótulo de churn ligeiramente diferentes.
- O dataset cobre apenas um ano; não é possível validar se o modelo generalizaria para anos seguintes ou para mudanças estruturais no negócio.

# 12. Comparação Final dos Modelos

A tabela abaixo resume o desempenho de todos os modelos avaliados neste MVP, do baseline ao modelo final otimizado.

In [ ]:
comparison = pd.DataFrame(results).T
comparison = comparison[["accuracy", "precision", "recall", "f1", "roc_auc", "train_time_s"]]
comparison = comparison.round(4)
comparison.loc[comparison.index] .style.highlight_max(subset=["f1", "roc_auc"], color="lightgreen")

**Leitura da tabela:**

| Modelo | Papel | Observação |
|---|---|---|
| `baseline` | referência mínima | sempre prevê "ativo"; F1 = 0 para a classe churn |
| `GradientBoosting` | candidato 1 (hiperparâmetros padrão) | supera claramente o baseline |
| `KNN` | candidato 2 (família diferente) | supera o baseline, mas fica abaixo do Gradient Boosting |
| `GradientBoosting_otimizado` | **modelo final** | melhor F1 e ROC-AUC após ajuste de hiperparâmetros |

O ganho do modelo otimizado sobre o `GradientBoosting` padrão é modesto, mas consistente — o que é esperado, já que o modelo padrão já era competitivo; a otimização refina o resultado em vez de transformá-lo radicalmente.

# 13. Boas Práticas e Rastreabilidade

- **Seed fixa:** `SEED = 42`, usada em todas as divisões de dados, inicializações de modelo e na busca de hiperparâmetros, garantindo que os resultados deste notebook sejam reproduzíveis.
- **Bibliotecas utilizadas:** `pandas`, `numpy`, `matplotlib`, `scikit-learn` e `scipy` — todas já disponíveis por padrão no Google Colab, sem necessidade de instalação adicional.
- **Tempo de treino:** reportado por modelo na coluna `train_time_s` da tabela comparativa (Seção 12); a busca de hiperparâmetros (Seção 10) foi a etapa mais custosa, mas ainda assim executada em menos de dois minutos em CPU padrão do Colab.
- **Recursos computacionais:** todo o notebook roda em CPU, sem necessidade de GPU, dado o tamanho moderado da base de clientes (~3 mil linhas na tabela final de features).

**Registro de decisões:**

| Decisão | Justificativa | Impacto esperado |
|---|---|---|
| Definir churn via corte temporal (9m observação / 3m futuro) | Simula um cenário real de uso do modelo e evita vazamento de dados | Rótulo confiável e sem contaminação do futuro nas features |
| Descartar transações sem `CustomerID` | Problema é definido no nível do cliente | Perda de ~25% das linhas, mas necessária |
| Transformação `log1p` em atributos de cauda longa | Reduz influência desproporcional de clientes atípicos (atacadistas) sem excluí-los | Modelos sensíveis à escala (KNN) generalizam melhor |
| Escolher F1-score/ROC-AUC como métricas principais | Classe-alvo moderadamente desbalanceada; acurácia isolada seria enganosa | Avaliação mais fiel ao objetivo de negócio |
| Otimizar hiperparâmetros do Gradient Boosting via `RandomizedSearchCV` | Melhor candidato inicial; espaço de busca contínuo | Ganho incremental de desempenho com custo computacional controlado |

# 14. Conclusão

Este MVP teve como objetivo construir e avaliar modelos de Machine Learning capazes de prever o **churn de clientes** de um varejista online (dataset **Online Retail**, UCI), a partir de atributos de comportamento de compra (RFM e derivados), comparando um baseline ingênuo com modelos candidatos de famílias distintas.

**Principais tratamentos realizados:** remoção de transações sem `CustomerID`, separação entre transações válidas e canceladas, construção de um corte temporal para gerar o rótulo `Churned` sem vazamento de dados, engenharia de atributos por cliente (recência, frequência, valor monetário e derivados), transformação logarítmica de atributos de cauda longa, padronização de variáveis numéricas e codificação one-hot da variável categórica `Country` — todas encapsuladas em um `Pipeline` ajustado apenas nos dados de treino.

**Modelos avaliados:** `DummyClassifier` (baseline), `GradientBoostingClassifier` e `KNeighborsClassifier` (candidatos), e uma versão otimizada do Gradient Boosting via `RandomizedSearchCV` com validação cruzada estratificada.

**Melhor resultado:** o **Gradient Boosting com hiperparâmetros otimizados** apresentou o melhor F1-score e ROC-AUC no conjunto de teste, superando claramente o baseline e os demais candidatos, sem sinais de overfitting severo.

**Por que essa foi a melhor solução:** o Gradient Boosting lidou melhor do que o KNN com as interações não lineares entre recência, frequência e valor monetário, e a otimização de hiperparâmetros trouxe um ganho adicional e consistente, sem comprometer a generalização do modelo (a diferença entre desempenho no treino e no teste permaneceu moderada).

**O MVP cumpriu o objetivo definido no início:** sim — foi construído um fluxo completo, documentado e reprodutível, que parte de dados transacionais brutos e chega a um modelo capaz de identificar clientes em risco de churn com desempenho consistentemente superior a uma regra ingênua.

**Limitações do projeto:**
- O rótulo de churn depende de uma escolha específica de corte temporal (9 meses / 3 meses), que não foi exaustivamente testada com outras janelas.
- O dataset cobre apenas 12 meses e é fortemente concentrado no Reino Unido, limitando a generalização geográfica e temporal.
- Não foram incorporadas informações externas (marketing, sazonalidade de longo prazo, concorrência).
- A busca de hiperparâmetros foi deliberadamente compacta (25 iterações), adequada ao escopo de um MVP, mas não exaustiva.

**Próximos passos sugeridos:**
- Testar diferentes janelas de corte temporal e validar a estabilidade do modelo ao longo do tempo (ex.: retreinar mensalmente).
- Incorporar novos atributos, como categorias de produtos mais comprados por cliente.
- Testar modelos adicionais (ex.: Regressão Logística com regularização para maior interpretabilidade, ou XGBoost/LightGBM para desempenho).
- Traduzir a saída do modelo em uma lista priorizada de clientes para campanhas de retenção, e medir o impacto real dessas campanhas (teste A/B) — fechando o ciclo de Machine Learning aplicado ao negócio.

# 15. Checklist do MVP (Autoavaliação)

**Definição do problema**
- *Qual é a descrição do problema?* Prever churn de clientes de um varejista online a partir do histórico de compras (Seção 1.1).
- *Qual é o objetivo do modelo?* Classificar clientes como "em risco de churn" ou "ativos" para priorizar ações de retenção (Seção 1.2).
- *O problema é de classificação, regressão, clusterização ou séries temporais?* Classificação binária (Seção 1.3).
- *Por que pode ser resolvido com ML?* Existe histórico comportamental preditivo e um rótulo observável construído a partir dos próprios dados (Seção 1.3).
- *Há premissas/hipóteses?* Sim, listadas na Seção 1.4 (recência, frequência e monetário associados ao churn).
- *Quais restrições foram consideradas na escolha dos dados?* Dataset não usado em aula, público, sem dados pessoais identificáveis (Seção 1.4).

**Descrição dos dados**
- *Qual dataset foi utilizado?* Online Retail (UCI). *Fonte?* https://archive.ics.uci.edu/dataset/352/online+retail (Seção 3.1).
- *Como os dados foram carregados?* Via URL pública (raw do GitHub), sem upload manual (Seção 3.2).
- *Quantos registros e atributos?* ~541 mil transações e 8 colunas originais (Seção 3.3); ~3.362 clientes e 10 atributos após engenharia de atributos (Seção 5.3).
- *Existe variável-alvo?* Sim, `Churned` (Seção 5.3).
- *Há limitações conhecidas?* Sim, listadas nas Seções 3.4 e 14.

**Preparação dos dados**
- *Houve valores ausentes? Como foram tratados?* Sim, em `CustomerID`; linhas removidas por não poderem ser atribuídas a um cliente (Seção 5.2).
- *Houve remoção/transformação de atributos?* Sim — cancelamentos isolados, `Country` simplificado, `log1p` em atributos de cauda longa (Seções 5.2, 5.5, 5.6).
- *Foram criados novos atributos?* Sim, todos os atributos RFM e derivados (Seção 5.3).
- *Normalização, padronização, encoding?* Sim, dentro do `Pipeline` (Seção 7).
- *Houve preocupação com vazamento de dados?* Sim — corte temporal para o rótulo (Seção 5.1) e ajuste das transformações apenas no treino (Seções 6 e 7).

**Divisão dos dados**
- *Como foi feita?* Treino/teste 80/20 estratificado (Seção 6).
- *Foi usada validação cruzada?* Sim, dentro da otimização de hiperparâmetros (Seção 10).
- *Séries temporais/clusterização se aplicam?* Não diretamente à divisão treino/teste (não é o tipo de problema), mas a ordem temporal foi respeitada na construção do rótulo (Seção 5.1).

**Modelagem**
- *Qual foi o baseline?* `DummyClassifier` (most_frequent) (Seção 8.1).
- *Quais modelos foram treinados?* Gradient Boosting e KNN, além do Gradient Boosting otimizado (Seções 8.2, 10).
- *Por que esses modelos?* Famílias distintas entre si e diferentes das já usadas no template de referência (Seção 8.2).
- *Foram comparados de forma justa?* Sim, mesmo pipeline de pré-processamento e mesma divisão treino/teste para todos.
- *Underfitting/overfitting?* Discutido na Seção 11.1 — sem sinais de underfitting; overfitting moderado e controlado.

**Otimização**
- *Algum modelo teve hiperparâmetros ajustados?* Sim, o Gradient Boosting (Seção 10).
- *Qual estratégia de busca?* `RandomizedSearchCV` com validação cruzada estratificada de 5 folds.
- *Trouxe melhora?* Sim, incremental (Seção 10.1).
- *Usou indevidamente o teste?* Não — a busca usou apenas validação cruzada sobre o treino.

**Avaliação**
- *Quais métricas?* Acurácia, precisão, recall, F1-score e ROC-AUC (Seção 9), com matriz de confusão e curva ROC para o modelo final (Seção 11).
- *Por que essas métricas?* Adequadas a um problema de classificação binária moderadamente desbalanceado (Seção 5.4).
- *Qual modelo teve melhor desempenho?* Gradient Boosting otimizado (Seções 11 e 12).
- *Houve análise de erros?* Sim (Seção 11.2).
- *Principais limitações?* Listadas nas Seções 11.2 e 14.

**Conclusão**
- *Qual foi a melhor solução?* Gradient Boosting com hiperparâmetros otimizados (Seção 14).
- *Por que foi escolhida?* Melhor F1-score e ROC-AUC, sem overfitting severo (Seção 14).
- *O MVP cumpriu o objetivo inicial?* Sim (Seção 14).
- *Quais são os próximos passos?* Listados na Seção 14.